# YOLO11s 베이스라인 학습

**전제:** 팀원 노트북(`pill_detection_dataset.ipynb`)을 실행해 `data/processed` 아래에
`images/{train,val,test}`, `labels/{train,val,test}`, `data.yaml` 이 생성돼 있어야 합니다.
이 노트북(`notebooks/` 안에 위치)은 그 `data.yaml`만 받아 YOLO11s를 학습합니다.

베이스라인은 팀 합의대로 **기본값 위주**로 돌립니다.


## 1. 설치 환경 재현성

In [1]:
# ============================================================
# 1. 설치, 환경, 재현성
# ============================================================

# Colab이면 실행 (로컬은 최초 1회만)
#!pip -q install ultralytics

import os, random, numpy as np, torch
from pathlib import Path
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| device:", DEVICE)

torch: 2.5.1+cu121 | CUDA: True | device: 0


## 2. 경로 설정

In [ ]:
# ============================================================
# 2-1. 구글 드라이브 마운트
# ============================================================
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/코드잇_파트2_3팀_프로젝트/pill-object-detection")

In [ ]:
from pathlib import Path
# ============================================================
# 2-2. 경로 설정 (모든 경로를 여기서 한 번에 관리)
# ============================================================
#PROJECT_ROOT = Path("/content/drive/MyDrive/코드잇 AI 13기/AI 13기 프로젝트/pill-object-detection/코드잇_파트2_3팀_프로젝트/pill-object-detection")
PROJECT_ROOT = Path("..")

DATASET_DIR     = PROJECT_ROOT / "data" / "processed"          # YOLO 데이터셋 (images/labels/data.yaml)
#DATASET_DIR     = PROJECT_ROOT / "data" / "processed_ver2_no_aug"
#TEST_IMAGE_DIR  = (PROJECT_ROOT / "data" / "dataset" / "cleaning_data"
#                  / "260813_improve_dataset_v2" / "sprint_ai_project1_data"/ "test_images")   # Kaggle 테스트 842장
TEST_IMAGE_DIR  = (PROJECT_ROOT / "data" / "dataset" / "cleaning_data" / "test_images")
#ANNOTATION_DIR = (PROJECT_ROOT / "data" / "dataset" / "cleaning_data"
#                  / "260813_improve_dataset_v2" / "sprint_ai_project1_data"/ "train_annotation_clean")
ANNOTATION_DIR = (PROJECT_ROOT / "data" / "dataset" / "cleaning_data" / "train_annotations")
WEIGHTS_PATH    = PROJECT_ROOT / "notebooks" / "runs" / "outputs" / "yolo" / "yolo11s_mix8" / "weights" / "best.pt"
SUBMISSION_PATH = PROJECT_ROOT / "notebooks" / "runs" / "outputs" / "submissions" / "submission.csv"

# 존재 확인
print("DATASET_DIR    exists:", DATASET_DIR.exists(), "(필수)")
print("TEST_IMAGE_DIR exists:", TEST_IMAGE_DIR.exists(), "(필수)")
print("ANNOTATION_DIR exists:", ANNOTATION_DIR.exists(), "(필수)")
print("WEIGHTS_PATH   exists:", WEIGHTS_PATH.exists(), "(학습 후 생성)")
print("SUBMISSION dir exists:", SUBMISSION_PATH.exists(), "(제출 시 생성)")

DATASET_DIR    exists: True (필수)
TEST_IMAGE_DIR exists: True (필수)
ANNOTATION_DIR exists: True (필수)
WEIGHTS_PATH   exists: False (학습 후 생성)
SUBMISSION dir exists: False (제출 시 생성)


## 3. data.yaml

In [7]:
# ==================================================================
# 3. data.yaml 경로 이식성 처리
# ==================================================================

import yaml
src_yaml = DATASET_DIR / "data.yaml"
cfg = yaml.safe_load(open(src_yaml, encoding="utf-8"))
cfg["path"] = str(DATASET_DIR.resolve())
yaml.safe_dump(cfg, open(src_yaml, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
RUNTIME_YAML = src_yaml          # 이후 코드는 그대로 RUNTIME_YAML 사용

assert (DATASET_DIR / "images" / "train").is_dir(), \
    "images/train 이 없습니다. pill_detection_dataset.ipynb 를 먼저 실행해 data/processed 를 만드세요."

print("클래스 수:", cfg["nc"], "| yaml:", RUNTIME_YAML)

클래스 수: 56 | yaml: ..\data\processed_ver2_no_aug\data.yaml


## 4. 학습 (W&B 로깅 포함)

In [5]:
# ==================================================================
# 4-1. W&B 설치 및 로그인
# ==================================================================
#!pip -q install wandb
import wandb
from ultralytics import settings

settings.update({"wandb": True})          # Ultralytics 내장 W&B 로깅 ON
wandb.login()   # 최초 1회, wandb.ai/authorize 의 API 키

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\soulf\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [6]:
# ============================================================
# 4-2. 베이스라인 학습 (YOLO11s) + W&B 로깅
# ============================================================

wandb.init(project="pill-object-detection", name="yolo11s",
           entity="diokim17",                       # ← 팀 entity (조직 -org 아님)
           job_type="train", tags=["yolo11s","mix8"], notes="yolo11s + mix8 + 50 epochs + patience 10 + hsv_h(0.015), hsv_s(0.3), hsv_v(0.4), translate(0.1), degree(90), shear(0.5), scale(0.0)")

model = YOLO("yolo11s.pt")
IMGSZ = 640
results = model.train(
    data=str(RUNTIME_YAML), epochs=50, patience=10,  # 10 epoch 동안 val loss 개선 없으면 조기 종료
    imgsz=IMGSZ, batch=16, # 메모리 부족하면 batch 8 또는 -1(자동)
    seed=SEED, deterministic=True, device=DEVICE, workers=2, #workers=0: colab에서 오류 방지? 숫자 올려서 실험해봐.
    project=str(PROJECT_ROOT / "outputs/yolo"),
    name="yolo11s_mix8", exist_ok=True,
    # ── mix 8: hsv_h(0.015), hsv_s(0.3), hsv_v(0.4), translate(0.1), degree(90), shear(0.5), scale(0.0) ──
    hsv_h=0.015, hsv_s=0.3, hsv_v=0.4,
    degrees=90.0, translate=0.1, scale=0.0, shear=0.5, perspective=0.0,
    flipud=0.0, fliplr=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
)
print("결과 폴더:", results.save_dir)

wandb: Currently logged in as: seolwoom (diokim17) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


NameError: name 'RUNTIME_YAML' is not defined

## 5. 검증 (mAP)


In [28]:
# ==================================================================
# 5. 검증 지표 (mAP)
# ==================================================================
m = model.val(data=str(RUNTIME_YAML), split="val", device=DEVICE)
print(f"최종 Best Validation mAP@0.5:0.95: {m.box.map:.4f}")
print(f"          mAP@0.5 : {m.box.map50:.4f}")
print(f"          mAP@0.75: {m.box.map75:.4f}")

Ultralytics 8.4.120  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4080, 16376MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 1917.6167.8 MB/s, size: 1714.3 KB)
val: Scanning C:\Users\soulf\Documents\codeit\pill-object-detection\data\processed\labels\val.cache... 348 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 348/348  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 4.1it/s 5.3s0.2s
                   all        348       1298      0.967      0.973      0.971      0.958
             5mg          6          6      0.987          1      0.995      0.995
            100mg          3          3      0.966          1      0.995      0.962
            2mg          8          8      0.989          1      0.995      0.995
    ()()          6          6          1          1      0.995      0.995
     ()()         32         32      0.968      0.944       0.96       0.96
(-)         65         65  

## 6. 예측 시각화

In [7]:
# ==================================================================
# 6. 예측 시각화 (검증 이미지)
# ==================================================================

val_imgs = sorted((DATASET_DIR / "images" / "val").glob("*.png"))[:6]
pred = model.predict(
    val_imgs, imgsz=640, conf=0.25, max_det=4, device=DEVICE,
    save=True, project=str(PROJECT_ROOT / "outputs/yolo"), name="pred_val", exist_ok=True,
)
print("시각화 저장 위치:", pred[0].save_dir)


0: 640x512 1 ()(), 1 (), 1  100/1000mg, 3.5ms
1: 640x512 1 8 650mg, 1  5mg, 1  500/20mg, 3.5ms
2: 640x512 1 , 1  100mg, 1 (), 3.5ms
3: 640x512 1 8 650mg, 1  300mg, 1 , 1  100mg, 3.5ms
4: 640x512 1  10mg, 1 , 1  100/1000mg, 3.5ms
5: 640x512 1 (), 1  20mg, 1 , 1  2.5/850mg, 3.5ms
Speed: 7.1ms preprocess, 3.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 512)
Results saved to C:\Users\soulf\Documents\codeit\pill-object-detection\notebooks\runs\outputs\yolo\pred_val
시각화 저장 위치: C:\Users\soulf\Documents\codeit\pill-object-detection\notebooks\runs\outputs\yolo\pred_val


## 7. 역매핑


아래 셀을 실행 시킨 후 8.제출 코드를 실행할 것. 

PillDetectionDataset.get_category_id(int)로 아래 주석처리된 역매핑용 to_category_id()를 대체함. 

In [1]:
import sys
from pathlib import Path

module_dir = Path("../src").resolve()
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from PillDetectionDataset import PillDetectionDataset

project_root = Path("..").resolve()

dataset_root = "../data/dataset/cleaning_data" #현재 git 폴더 구조에 맞도록 변경

dataset = PillDetectionDataset(
    root=dataset_root,
    image_dir_name="train_images",
    annotation_dir_name="train_annotations",
    label_offset=0,
    strict=False,
)

print(dataset.get_category_id(0))  # 클래스 ID 0에 대한 카테고리 정보 가져오기. 1900이 나오면 성공

1900


In [ ]:
# ============================================================
# 7. YOLO 클래스(0~55) → 원본 category_id 역매핑
# ============================================================
# import json

# def build_yolo_to_category_id(annotation_dir: Path) -> dict:
#     category_ids = set()
#     for json_path in annotation_dir.rglob("*.json"):
#         with json_path.open("r", encoding="utf-8") as f:
#             data = json.load(f)
#         for cat in data.get("categories", []):
#             category_ids.add(int(cat["id"]))
#     # pill_detection 파이프라인과 동일하게: 정렬 후 0부터 인덱싱
#     sorted_ids = sorted(category_ids)
#     return {yolo_id: cid for yolo_id, cid in enumerate(sorted_ids)}

# yolo2cat = build_yolo_to_category_id(ANNOTATION_DIR)
# assert len(yolo2cat) == 56, f"클래스 수 이상: {len(yolo2cat)}개"

# def to_category_id(yolo_cls: int) -> int:
#     return yolo2cat[yolo_cls]      # 예: 3 → 58873

# print("클래스 수:", len(yolo2cat))
# print("앞 5개 (yolo_id → category_id):", list(yolo2cat.items())[:5])

AssertionError: 클래스 수 이상: 57개

## 8. 제출

In [ ]:
# ============================================================
# 8. Kaggle 제출 파일 생성
# ============================================================
import csv, gc
from PIL import Image

model = YOLO(str(WEIGHTS_PATH))          # best.pt 로드 (세션 재시작 후 추론만도 가능)

test_files = sorted(TEST_IMAGE_DIR.glob("*.png"), key=lambda p: int(p.stem))
print("테스트 이미지:", len(test_files), "| 첫 장 크기:", Image.open(test_files[0]).size)

# 제출 전 1장 sanity 체크
r0 = model.predict(str(test_files[0]), imgsz=IMGSZ, conf=0.25, max_det=4, device=DEVICE, verbose=False)[0]
print("첫 이미지:", test_files[0].name, "| 검출 수:", len(r0.boxes),
      "| category_id 예:", [dataset.get_category_id(int(c)) for c in r0.boxes.cls.cpu().numpy()])

테스트 이미지: 842 | 첫 장 크기: (976, 1280)
첫 이미지: 1.png | 검출 수: 4 | category_id 예: [27926, 16551, 24850, 1900]


In [ ]:
# ============================================================
# 8. Kaggle 제출 파일 생성 - continued
# ============================================================

rows, ann_id = [], 1
for i, f in enumerate(test_files):
    r = model.predict(str(f), imgsz=IMGSZ, conf=0.25, max_det=4, device=DEVICE, verbose=False)[0]
    image_id = int(f.stem)
    b = r.boxes
    for xyxy, conf, cls in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
        x1, y1, x2, y2 = xyxy
        rows.append([ann_id, image_id, dataset.get_category_id(int(cls)),
                     int(round(x1)), int(round(y1)),
                     int(round(x2 - x1)), int(round(y2 - y1)), round(float(conf), 4)])
        ann_id += 1
    del r
    if i % 100 == 0:
        gc.collect(); print(f"{i}/{len(test_files)} 처리 중...")

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as file:
    w = csv.writer(file)
    w.writerow(["annotation_id","image_id","category_id","bbox_x","bbox_y","bbox_w","bbox_h","score"])
    w.writerows(rows)
print("작성 완료:", SUBMISSION_PATH, "| 행:", len(rows))

0/842 처리 중...
100/842 처리 중...
200/842 처리 중...
300/842 처리 중...
400/842 처리 중...
500/842 처리 중...
600/842 처리 중...
700/842 처리 중...
800/842 처리 중...
작성 완료: ..\notebooks\runs\outputs\submissions\submission.csv | 행: 3229


In [8]:
# ============================================================
# 8. 저장한 Kaggle 제출 파일을 다시 읽어 확인
# ============================================================
import pandas as pd
df = pd.read_csv(SUBMISSION_PATH)
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("annotation_id 고유:", df["annotation_id"].is_unique)
print("image_id 수:", df["image_id"].nunique())                       # 검출된 이미지 수
print("category_id 범위:", df["category_id"].min(), "~", df["category_id"].max())
print("category_id 예시:", sorted(df["category_id"].unique())[:10])   # 5자리 코드여야 함
print("score 범위:", round(df["score"].min(),4), "~", round(df["score"].max(),4))

shape: (3229, 8)
columns: ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']
annotation_id 고유: True
image_id 수: 842
category_id 범위: 1900 ~ 44199
category_id 예시: [np.int64(1900), np.int64(2483), np.int64(3351), np.int64(3483), np.int64(3544), np.int64(3743), np.int64(3832), np.int64(4378), np.int64(4543), np.int64(5094)]
score 범위: 0.8538 ~ 0.997


## 9. eval 셋으로 mAP/IoU 평가

In [33]:
# ============================================================
# 9. eval(test) split 성능 평가 — 라벨이 있어 로컬에서 mAP/IoU 계산 가능
# ============================================================
model = YOLO(str(WEIGHTS_PATH))          # 학습된 가중치
m_eval = model.val(data=str(RUNTIME_YAML), split="test", device=DEVICE,
                   plots=True, project=str(PROJECT_ROOT / "outputs/yolo"), name="eval_test", exist_ok=True)

print(f"[eval] mAP@0.5:0.95 : {m_eval.box.map:.4f}")   # IoU 0.5~0.95 평균
print(f"[eval] mAP@0.5      : {m_eval.box.map50:.4f}")  # IoU 0.5 기준
print(f"[eval] mAP@0.75     : {m_eval.box.map75:.4f}")  # IoU 0.75 기준
print("결과·그래프 저장:", m_eval.save_dir)

Ultralytics 8.4.120  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4080, 16376MiB)
YOLO11s summary (fused): 101 layers, 9,450,726 parameters, 0 gradients, 21.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1573.8222.5 MB/s, size: 1764.3 KB)
val: Scanning C:\Users\soulf\Documents\codeit\pill-object-detection\data\processed\labels\test.cache... 325 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 325/325  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 3.3it/s 6.3s0.2s
                   all        325       1197      0.991      0.994      0.994      0.983
            2mg         19         19      0.996          1      0.995      0.975
     ()()         25         25      0.993          1      0.995      0.995
(-)         61         61      0.998          1      0.995      0.995
                           47         47      0.998      0.979      0.987      0.987
          ()         33      

In [ ]:
# ============================================================
# 9-2. eval 이미지: 정답 vs 예측 나란히 시각화
# ============================================================
# 한글 폰트 (Colab)
!apt-get -qq install -y fonts-nanum >/dev/null 2>&1
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from matplotlib.patches import Rectangle
from PIL import Image
for fp in fm.findSystemFonts():
    if "Nanum" in fp: fm.fontManager.addfont(fp)
plt.rcParams["font.family"] = "NanumGothic"; plt.rcParams["axes.unicode_minus"] = False

names = model.names   # {0: '보령부스파정 5mg', ...}
img_dir = DATASET_DIR / "images" / "test"
lbl_dir = DATASET_DIR / "labels" / "test"

def gt_boxes(stem, W, H):
    p = lbl_dir / f"{stem}.txt"
    out = []
    if p.exists():
        for line in p.read_text().strip().splitlines():
            c, cx, cy, w, h = map(float, line.split())
            out.append((int(c), (cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H))
    return out

eval_imgs = sorted(img_dir.glob("*.png"))[:4]
fig, axes = plt.subplots(len(eval_imgs), 2, figsize=(12, 5*len(eval_imgs)))
for row, ip in enumerate(eval_imgs):
    img = Image.open(ip).convert("RGB"); W, H = img.size
    # 정답
    ax = axes[row, 0]; ax.imshow(img); ax.set_title(f"정답 GT: {ip.name}"); ax.axis("off")
    for c, x1, y1, x2, y2 in gt_boxes(ip.stem, W, H):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", lw=2))
        ax.text(x1, y1-5, names[c], color="lime", fontsize=8)
    # 예측
    r = model.predict(str(ip), imgsz=640, conf=0.05, max_det=4, device=DEVICE, verbose=False)[0]
    ax = axes[row, 1]; ax.imshow(img); ax.set_title("예측 Pred"); ax.axis("off")
    for (x1,y1,x2,y2), c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", lw=2))
        ax.text(x1, y1-5, f"{names[int(c)]} {s:.2f}", color="red", fontsize=8)
plt.tight_layout(); plt.show()

## 10. submission.csv 예측 결과물 시각화

(아래 코드는 민협님 작성 노트북 파일인 pill_detection_dataset.ipynb 의 마지막 부분을 가져와 수정한 것)

이미지 842장 기준 8분 정도 소요

이미지를 영구보관 하거나 / 드라이브에 폴더를 만들어 다운받고 싶은 경우,  

`# output_dir = PROJECT_ROOT / "outputs" / "pred_visualizations"`

 을 uncomment 하면 됨

In [ ]:
from pathlib import Path
import math, json
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image
import pandas as pd
from urllib.request import Request, urlopen
import tempfile, shutil

# ============================================================
# 1. 한글 폰트 설정
# ============================================================
FONT_URL = ("https://raw.githubusercontent.com/google/fonts/"
            "main/ofl/nanumgothic/NanumGothic-Regular.ttf")
FONT_DIR = Path.home() / ".cache" / "matplotlib-korean-font"
FONT_PATH = FONT_DIR / "NanumGothic-Regular.ttf"

def download_font_if_needed():
    if FONT_PATH.exists() and FONT_PATH.stat().st_size > 0:
        return FONT_PATH
    FONT_DIR.mkdir(parents=True, exist_ok=True)
    req = Request(FONT_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=30) as resp:
        with tempfile.NamedTemporaryFile(dir=FONT_DIR, suffix=".tmp", delete=False) as tf:
            shutil.copyfileobj(resp, tf); tmp = Path(tf.name)
    tmp.replace(FONT_PATH); return FONT_PATH

try:
    fp = download_font_if_needed()
    fm.fontManager.addfont(str(fp))
    font_property = fm.FontProperties(fname=str(fp))
    plt.rcParams["font.family"] = font_property.get_name()
    plt.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 준비 실패:", e); font_property = None

# ============================================================
# 2. 시각화 설정
# ============================================================
images_per_page, num_cols = 10, 5
num_rows = math.ceil(images_per_page / num_cols)
save_figures, show_figures = True, False
# output_dir = PROJECT_ROOT / "outputs" / "pred_visualizations"
output_dir = Path("./pred_visualizations")
output_dir.mkdir(parents=True, exist_ok=True)
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# ============================================================
# 3. category_id → 약 이름 매핑
# ============================================================
cat2name = {}
for jf in ANNOTATION_DIR.rglob("*.json"):
    for c in json.load(open(jf, encoding="utf-8")).get("categories", []):
        cat2name[int(c["id"])] = c["name"]

def find_image(image_id):                       # 파일명 형식 자동 탐색
    for pat in (f"{image_id}.png", f"{image_id}.jpg", f"image{image_id}.png"):
        p = TEST_IMAGE_DIR / pat
        if p.exists(): return p
    return TEST_IMAGE_DIR / f"{image_id}.png"

# ============================================================
# 4. submission.csv 예측 시각화  (★ image_id 단위로 그룹)
# ============================================================
sub = pd.read_csv(SUBMISSION_PATH)
image_ids = sorted(sub["image_id"].unique())    # ★ annotation_id 아님! image_id로 묶음
# image_ids = image_ids[:30]                    # ← 먼저 일부만 미리보려면 주석 해제

num_images = len(image_ids)
num_pages = math.ceil(num_images / images_per_page)
print(f"전체 이미지 수: {num_images}\n생성할 페이지 수: {num_pages}")

for page_index in range(num_pages):
    start, end = page_index*images_per_page, min((page_index+1)*images_per_page, num_images)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(25, 11), constrained_layout=True)
    axes = np.asarray(axes).reshape(-1)

    for sub_i, image_id in enumerate(image_ids[start:end]):
        ax = axes[sub_i]
        img_path = find_image(image_id)
        try:
            ax.imshow(Image.open(img_path).convert("RGB"))
        except FileNotFoundError:
            ax.text(0.5, 0.5, f"{img_path.name}\n없음", ha="center"); ax.axis("off"); continue

        dets = sub[sub["image_id"] == image_id]          # ★ 그 이미지의 모든 검출(행)
        for j, (_, r) in enumerate(dets.iterrows()):
            x, y, w, h = float(r["bbox_x"]), float(r["bbox_y"]), float(r["bbox_w"]), float(r["bbox_h"])
            color = colors[j % len(colors)]
            ax.add_patch(Rectangle((x, y), w, h, linewidth=2.5, edgecolor=color, facecolor="none"))
            name = cat2name.get(int(r["category_id"]), "unknown")
            text = f"{name}\nID: {int(r['category_id'])}\nscore: {r['score']:.2f}"
            ty, va = (y - 5, "bottom") if y >= 70 else (y + 5, "top")
            ax.text(x, ty, text, fontsize=8, color="white", verticalalignment=va,
                    fontproperties=font_property,
                    bbox={"facecolor": color, "alpha": 0.85, "edgecolor": "none", "pad": 2})

        ax.set_title(f"[{image_id}] {img_path.name}\n검출 수: {len(dets)}",
                     fontsize=9, fontproperties=font_property)
        ax.axis("off")

    for k in range(end - start, len(axes)):
        axes[k].axis("off")

    fig.suptitle(f"YOLO 예측 시각화 ({start+1}–{end} / {num_images})",
                 fontsize=18, fontproperties=font_property)

    if save_figures:
        out = output_dir / f"pred_page_{page_index+1:03d}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
        if (page_index+1) % 10 == 0 or page_index+1 == num_pages:
            print(f"[{page_index+1:03d}/{num_pages:03d}] 저장 완료")
    if show_figures:
        plt.show()
    plt.close(fig)

print(f"\n전체 시각화 완료: {output_dir.resolve()}")

In [ ]:
from IPython.display import Image as IPyImage, display
from pathlib import Path

pages = sorted(Path("/content/pred_visualizations").glob("*.png"))
print(f"{len(pages)}장 표시")
for p in pages:                 # 일부만 보려면 pages[:10]
    display(IPyImage(filename=str(p), width=1400))